In [3]:
import pandas as pd
import numpy as np


In [4]:
df=pd.read_csv('data/tested.csv')

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  418 non-null    int64  
 1   Survived     418 non-null    int64  
 2   Pclass       418 non-null    int64  
 3   Name         418 non-null    object 
 4   Sex          418 non-null    object 
 5   Age          332 non-null    float64
 6   SibSp        418 non-null    int64  
 7   Parch        418 non-null    int64  
 8   Ticket       418 non-null    object 
 9   Fare         417 non-null    float64
 10  Cabin        91 non-null     object 
 11  Embarked     418 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 39.3+ KB


In [6]:
df.drop(columns=['PassengerId','Name','Cabin','Fare','Ticket'],inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Survived  418 non-null    int64  
 1   Pclass    418 non-null    int64  
 2   Sex       418 non-null    object 
 3   Age       332 non-null    float64
 4   SibSp     418 non-null    int64  
 5   Parch     418 non-null    int64  
 6   Embarked  418 non-null    object 
dtypes: float64(1), int64(4), object(2)
memory usage: 23.0+ KB


In [7]:
from sklearn.model_selection import train_test_split
xTrain,xTest,yTrain,yTest=train_test_split(df.drop(columns=['Survived']),df['Survived'],test_size=0.2,random_state=0)

In [8]:
xTrain.shape


(334, 6)

In [9]:
xTrain.isnull().sum()
xTrain.head()

,Pclass,Sex,Age,SibSp,Parch,Embarked
20,1,male,55.0,1,0,C
306,1,male,30.0,1,2,S
142,1,male,61.0,1,3,C
14,1,female,47.0,1,0,S
284,3,female,2.0,1,1,S


In [10]:
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

ct1=ColumnTransformer([
    ('age',SimpleImputer(strategy='mean'),[2])
],remainder='passthrough')

In [11]:
from sklearn.preprocessing import OneHotEncoder

ct2=ColumnTransformer([
    ('ohe_sex_embark',OneHotEncoder(sparse_output=False,handle_unknown='ignore'),[3,5])
],remainder='passthrough')

In [12]:
from sklearn.preprocessing import StandardScaler

ct3=ColumnTransformer([
    ('scale',StandardScaler(),slice(0,9))
])

In [13]:
from sklearn.tree import DecisionTreeClassifier

dct=DecisionTreeClassifier()


In [14]:
from sklearn.pipeline import Pipeline

pipe=Pipeline([
    ('ct1',ct1),
    ('ct2',ct2),
    ('ct3',ct3),
    ('dct',dct),
])

In [15]:
pipe.fit(xTrain,yTrain)

,steps,"[('ct1', ...), ('ct2', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('age', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [16]:
pipe.named_steps

{'ct1': ColumnTransformer(remainder='passthrough',
                   transformers=[('age', SimpleImputer(), [2])]),
 'ct2': ColumnTransformer(remainder='passthrough',
                   transformers=[('ohe_sex_embark',
                                  OneHotEncoder(handle_unknown='ignore',
                                                sparse_output=False),
                                  [3, 5])]),
 'ct3': ColumnTransformer(transformers=[('scale', StandardScaler(), slice(0, 9, None))]),
 'dct': DecisionTreeClassifier()}

In [17]:
prediction=pipe.predict(xTest)
prediction

array([0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0,
       0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0,
       0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0,
       0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1])

In [18]:
from sklearn.metrics import accuracy_score
accuracy_score(yTest,prediction)

0.5119047619047619

In [19]:
from sklearn.model_selection import cross_val_score
cross_val_score(pipe, xTrain, yTrain, cv=5, scoring='accuracy').mean()

np.float64(0.6288557213930348)